# 03 - Exploratory Data Analysis (Merged CICIDS2017 Dataset)

EDA on the cleaned, merged dataset (`02_data_cleaning.ipynb` output) covering the full CICIDS2017 week — 15 attack classes instead of the single-day subset the old notebook used.

Two things carried forward from earlier findings that shape this EDA:
- **Severe class imbalance across 15 classes** (not just a BENIGN-vs-attack split) — some classes like Heartbleed have only a handful of rows in the entire week.
- **Web Attack subclasses** (Brute Force / XSS / SQL Injection) are small and easily confused — this is why the project later needed a dedicated SMOTE-oversampled submodel and confidence-based uncertainty flagging for ambiguous Brute Force vs. XSS predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully!")

In [ ]:
df = pd.read_parquet("../datasets/processed/cicids2017_cleaned.parquet", engine="pyarrow")

print("Cleaned Merged Dataset Loaded Successfully!")

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Rows              : {df.shape[0]}")
print(f"Columns           : {df.shape[1]}")
print(f"Attack Classes    : {df['Label'].nunique()}")

print("=" * 60)

## Label distribution

In [ ]:
df["Label"].value_counts()

In [ ]:
round(df["Label"].value_counts(normalize=True) * 100, 3)

## Visualizing the class imbalance

Log scale on the count plot, since BENIGN and the largest attack classes (DoS Hulk, PortScan, DDoS) dwarf rare classes like Heartbleed and Infiltration by several orders of magnitude. A linear-scale plot would make the rare classes invisible.

In [ ]:
plt.figure(figsize=(14, 7))

order = df["Label"].value_counts().index

sns.countplot(data=df, x="Label", order=order)

plt.yscale("log")
plt.xticks(rotation=60, ha="right")
plt.title("Attack Distribution (log scale)")
plt.xlabel("Attack Type")
plt.ylabel("Count (log scale)")
plt.tight_layout()
plt.show()

With 15 classes a pie chart of everything is unreadable, so the top classes are shown individually and the long tail of rare attack types is grouped into `Other`.

In [ ]:
top_n = 7

label_counts = df["Label"].value_counts()

top_labels = label_counts.head(top_n)

other_total = label_counts.iloc[top_n:].sum()

pie_data = pd.concat([top_labels, pd.Series({"Other": other_total})])

plt.figure(figsize=(9, 9))

pie_data.plot(kind="pie", autopct="%1.1f%%")

plt.ylabel("")
plt.title(f"Attack Distribution (Top {top_n} Classes + Other)")
plt.show()

## Numeric feature overview

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

numeric_df.head()

## Correlation matrix (sampled)

Same sampling approach as the data understanding notebook -- computing `.corr()` on the full multi-million-row dataset is unnecessarily slow for what's meant to be a quick visual check.

In [ ]:
sample_df = df.sample(n=min(200_000, len(df)), random_state=42)

numeric_sample = sample_df.select_dtypes(include=np.number)

corr = numeric_sample.corr()

plt.figure(figsize=(20, 18))

sns.heatmap(corr, cmap="coolwarm", center=0)

plt.title("Correlation Matrix (200K-row sample)")
plt.show()

## Feature distributions (sampled)

Histograms across all ~78 numeric columns on the full dataset would take a long time and mostly show noise from rare classes -- a sample keeps this responsive.

In [ ]:
sample_df.select_dtypes(include=np.number).hist(
    figsize=(25, 25),
    bins=30
)

plt.show()

## Key features -- boxplots and distributions

In [ ]:
important_features = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Total Fwd Packets",
    "Total Backward Packets"
]

important_features = [f for f in important_features if f in df.columns]

for feature in important_features:

    plt.figure(figsize=(10, 4))

    sns.boxplot(x=sample_df[feature])

    plt.title(feature)

    plt.show()

In [ ]:
for feature in important_features:

    plt.figure(figsize=(8, 4))

    sns.histplot(sample_df[feature], kde=True)

    plt.title(feature)

    plt.show()

## Key features by attack class

Useful for a quick sanity check before modeling -- e.g. attack classes tend to show very different `Flow Duration` / `Flow Bytes/s` profiles from BENIGN traffic.

In [ ]:
df.groupby("Label")[important_features].mean().round(2)

## Web Attack subclasses

Flagging this explicitly since it drove a real modeling decision later in the project: the `Web Attack – Brute Force`, `Web Attack – XSS`, and `Web Attack – Sql Injection` classes are small and get confused with each other -- SQL Injection in particular is scarce. This is the reason a separate SMOTE-oversampled submodel (with 5-fold stratified CV) was built for Web Attack traffic, plus a confidence-based flag for ambiguous Brute Force vs. XSS calls.

In [ ]:
web_attack_labels = [l for l in df["Label"].unique() if "Web Attack" in str(l)]

print("Web Attack subclasses found:", web_attack_labels)

df[df["Label"].isin(web_attack_labels)]["Label"].value_counts()

## Summary

In [ ]:
print("=" * 60)

print("EDA SUMMARY")

print("=" * 60)

print(f"Total Records  : {df.shape[0]}")

print(f"Total Features : {df.shape[1]}")

print(f"Attack Classes : {df['Label'].nunique()}")

print()

print(df["Label"].value_counts())

print("=" * 60)